### Load Dataset
Load the raw retail inventory dataset for analysis.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/retail_store_inventory.csv")

df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


### Check Dataset Structure
Check the number of rows, columns, and available fields in the dataset.

In [2]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (73100, 15)

Columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 73100 entries, 0 to 73099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                73100 non-null  str    
 1   Store ID            73100 non-null  str    
 2   Product ID          73100 non-null  str    
 3   Category            73100 non-null  str    
 4   Region              73100 non-null  str    
 5   Inventory Level     73100 non-null  int64  
 6   Units Sold          73100 non-null  int64  
 7   Units Ordered       73100 non-null  int64  
 8   Demand Forecast     73100 non-null  float64
 9   Price               73100 non-null  float64
 10  Discount            73100 non-null  int64  
 11  Weather Condition   73100 non-null  str    
 12  Holiday/Promotion   73100 non-null  int64  
 13  Competitor Pricing  73100 non-null  float64
 14  Seasonality         73100 non-null  str    
dtypes: float64(3), int64(5), str(7)
memory usage: 11.4 MB


### Check Missing Values
Check whether any columns contain missing values.

In [4]:
df.isnull().sum()

Date                  0
Store ID              0
Product ID            0
Category              0
Region                0
Inventory Level       0
Units Sold            0
Units Ordered         0
Demand Forecast       0
Price                 0
Discount              0
Weather Condition     0
Holiday/Promotion     0
Competitor Pricing    0
Seasonality           0
dtype: int64

### Check Duplicate Rows
Check for exact duplicate records in the dataset.

In [5]:
df.duplicated().sum()

np.int64(0)

### Convert Date Column
Convert the `Date` column from text to datetime format for time-series analysis and forecasting.

In [6]:
df["Date"] = pd.to_datetime(df["Date"])

print(df["Date"].dtype)
print(df["Date"].min())
print(df["Date"].max())

datetime64[us]
2022-01-01 00:00:00
2024-01-01 00:00:00


### Check Numeric Value Ranges

Check numeric columns for negative or unusual values that may indicate data-quality issues.

In [7]:
numeric_cols = [
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand Forecast",
    "Price",
    "Discount",
    "Holiday/Promotion",
    "Competitor Pricing"
]

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Inventory Level,73100.0,274.469877,129.949514,50.00,162.00,273.000,387.0000,500.00
Units Sold,73100.0,136.464870,108.919406,0.00,49.00,107.000,203.0000,499.00
Units Ordered,73100.0,110.004473,52.277448,20.00,65.00,110.000,155.0000,200.00
Demand Forecast,73100.0,141.494720,109.254076,-9.99,53.67,113.015,208.0525,518.55
Price,73100.0,55.135108,26.021945,10.00,32.65,55.050,77.8600,100.00
Discount,73100.0,10.009508,7.083746,0.00,5.00,10.000,15.0000,20.00
Holiday/Promotion,73100.0,0.497305,0.499996,0.00,0.00,0.000,1.0000,1.00
Competitor Pricing,73100.0,55.146077,26.191408,5.03,32.68,55.010,77.8200,104.94


In [8]:
checks = {
    "Inventory Level < 0": (df["Inventory Level"] < 0).sum(),
    "Units Sold < 0": (df["Units Sold"] < 0).sum(),
    "Units Ordered < 0": (df["Units Ordered"] < 0).sum(),
    "Demand Forecast < 0": (df["Demand Forecast"] < 0).sum(),
    "Price < 0": (df["Price"] < 0).sum(),
    "Discount < 0": (df["Discount"] < 0).sum(),
    "Competitor Pricing < 0": (df["Competitor Pricing"] < 0).sum()
}

checks

{'Inventory Level < 0': np.int64(0),
 'Units Sold < 0': np.int64(0),
 'Units Ordered < 0': np.int64(0),
 'Demand Forecast < 0': np.int64(673),
 'Price < 0': np.int64(0),
 'Discount < 0': np.int64(0),
 'Competitor Pricing < 0': np.int64(0)}

### Inspect Negative Demand Forecasts
The `Demand Forecast` column contains negative values. Since demand cannot be negative, these records are treated as a data-quality issue and will be reviewed before applying any cleaning rule.

In [9]:
negative_forecast = df[df["Demand Forecast"] < 0]

negative_forecast[["Date", "Store ID", "Product ID", "Units Sold", "Demand Forecast"]].head(10)

,Date,Store ID,Product ID,Units Sold,Demand Forecast
63,2022-01-01,S004,P0004,0,-2.40
141,2022-01-02,S003,P0002,2,-3.40
278,2022-01-03,S004,P0019,1,-3.91
511,2022-01-06,S001,P0012,1,-8.37
730,2022-01-08,S002,P0011,7,-2.99
844,2022-01-09,S003,P0005,4,-1.33
1011,2022-01-11,S001,P0012,3,-6.05
1107,2022-01-12,S001,P0008,0,-3.55
1149,2022-01-12,S003,P0010,6,-0.11
1180,2022-01-12,S005,P0001,6,-2.04


### Quantify Negative Demand Forecasts

Measure the proportion of negative values in `Demand Forecast` before deciding how to handle them.

In [10]:
negative_count = (df["Demand Forecast"] < 0).sum()
total_count = len(df)

negative_percentage = (negative_count / total_count) * 100

print("Negative forecast records:", negative_count)
print("Total records:", total_count)
print(f"Negative forecast percentage: {negative_percentage:.2f}%")

Negative forecast records: 673
Total records: 73100
Negative forecast percentage: 0.92%


### Handle Invalid Demand Forecasts

In [11]:
df["Demand Forecast"] = df["Demand Forecast"].mask(
    df["Demand Forecast"] < 0
)

print("Missing Demand Forecasts after cleaning:",
      df["Demand Forecast"].isna().sum())

Missing Demand Forecasts after cleaning: 673


### Check Categorical Values

Review the unique values in key categorical columns to identify unexpected categories or inconsistent labels.

In [12]:
categorical_cols = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality"
]

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].unique())


Store ID:
<ArrowStringArray>
['S001', 'S002', 'S003', 'S004', 'S005']
Length: 5, dtype: str

Product ID:
<ArrowStringArray>
['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008',
 'P0009', 'P0010', 'P0011', 'P0012', 'P0013', 'P0014', 'P0015', 'P0016',
 'P0017', 'P0018', 'P0019', 'P0020']
Length: 20, dtype: str

Category:
<ArrowStringArray>
['Groceries', 'Toys', 'Electronics', 'Furniture', 'Clothing']
Length: 5, dtype: str

Region:
<ArrowStringArray>
['North', 'South', 'West', 'East']
Length: 4, dtype: str

Weather Condition:
<ArrowStringArray>
['Rainy', 'Sunny', 'Cloudy', 'Snowy']
Length: 4, dtype: str

Seasonality:
<ArrowStringArray>
['Autumn', 'Summer', 'Winter', 'Spring']
Length: 4, dtype: str


### Check Weather and Seasonality Values

Verify the unique values of weather conditions and seasonal labels to ensure consistent categorical data.

In [13]:
print("Weather Conditions:")
print(df["Weather Condition"].value_counts())

print("\nSeasonality:")
print(df["Seasonality"].value_counts())

Weather Conditions:
Weather Condition
Sunny     18290
Rainy     18278
Snowy     18272
Cloudy    18260
Name: count, dtype: int64

Seasonality:
Seasonality
Spring    18317
Summer    18305
Winter    18285
Autumn    18193
Name: count, dtype: int64


### Check Promotion and Discount Values

Review promotion and discount fields to verify that their values are within the expected range and consistently encoded.

In [14]:
print("Holiday/Promotion values:")
print(df["Holiday/Promotion"].value_counts().sort_index())

print("\nDiscount values:")
print(df["Discount"].value_counts().sort_index())

Holiday/Promotion values:
Holiday/Promotion
0    36747
1    36353
Name: count, dtype: int64

Discount values:
Discount
0     14662
5     14591
10    14508
15    14624
20    14715
Name: count, dtype: int64


### Check Weather and Seasonality Values

Verify the unique values of weather conditions and seasonal labels to ensure consistent categorical data.

In [15]:
print("Weather Conditions:")
print(df["Weather Condition"].value_counts())

print("\nSeasonality:")
print(df["Seasonality"].value_counts())

Weather Conditions:
Weather Condition
Sunny     18290
Rainy     18278
Snowy     18272
Cloudy    18260
Name: count, dtype: int64

Seasonality:
Seasonality
Spring    18317
Summer    18305
Winter    18285
Autumn    18193
Name: count, dtype: int64


### Check Daily Data Coverage

Verify that the dataset has consistent daily observations without unexpected gaps in the date coverage.

In [16]:
daily_counts = df.groupby("Date").size()

print("Number of unique dates:", daily_counts.size)
print("Minimum records per day:", daily_counts.min())
print("Maximum records per day:", daily_counts.max())

Number of unique dates: 731
Minimum records per day: 100
Maximum records per day: 100


### Check Store-Product Coverage

Verify that each store has complete observations for every product across the dataset.

In [17]:
store_product_counts = df.groupby(["Store ID", "Product ID"]).size()

print("Total Store-Product combinations:", store_product_counts.size)
print("Minimum records per combination:", store_product_counts.min())
print("Maximum records per combination:", store_product_counts.max())

Total Store-Product combinations: 100
Minimum records per combination: 731
Maximum records per combination: 731


### Analyze Sales Distribution

Examine the distribution of daily units sold, including zero-sales observations, to understand demand behavior before forecasting.

In [18]:
print("Zero-sales records:", (df["Units Sold"] == 0).sum())
print("Total records:", len(df))
print(f"Zero-sales percentage: {(df['Units Sold'] == 0).mean() * 100:.2f}%")

print("\nUnits Sold statistics:")
print(df["Units Sold"].describe())

Zero-sales records: 360
Total records: 73100
Zero-sales percentage: 0.49%

Units Sold statistics:
count    73100.000000
mean       136.464870
std        108.919406
min          0.000000
25%         49.000000
50%        107.000000
75%        203.000000
max        499.000000
Name: Units Sold, dtype: float64


### Analyze Product-Level Demand

Compare total units sold across products to identify high-demand and low-demand products.

In [19]:
product_sales = (
    df.groupby("Product ID")["Units Sold"]
      .sum()
      .sort_values(ascending=False)
)

product_sales

Product ID
P0016    508472
P0020    507708
P0014    507622
P0015    507283
P0005    503648
P0009    502086
P0013    500619
P0017    500510
P0011    499362
P0007    499321
P0001    498061
P0019    497899
P0006    497131
P0010    496469
P0004    495501
P0003    493279
P0018    492551
P0012    491670
P0008    488563
P0002    487827
Name: Units Sold, dtype: int64

### Product-Level Demand

Total units sold were calculated for each product to compare demand levels and identify high- and low-demand products for further EDA.

## 4-Table Data Model

The raw retail inventory dataset will be transformed into four logical tables required for Project FORESIGHT: sales_daily, sku_master, calendar, and inventory_snapshots.

The mapping will use only fields available in the selected dataset. Fields that are not directly available will be handled through documented assumptions rather than fabricated values.

In [20]:
table_mapping = {
    "sales_daily": [
        "Date",
        "Product ID",
        "Units Sold",
        "Price",
        "Discount",
        "Holiday/Promotion"
    ],
    
    "sku_master": [
        "Product ID",
        "Category"
    ],
    
    "calendar": [
        "Date",
        "Seasonality",
        "Holiday/Promotion"
    ],
    
    "inventory_snapshots": [
        "Date",
        "Product ID",
        "Inventory Level",
        "Units Ordered"
    ]
}

for table, columns in table_mapping.items():
    print(f"\n{table}:")
    print(columns)


sales_daily:
['Date', 'Product ID', 'Units Sold', 'Price', 'Discount', 'Holiday/Promotion']

sku_master:
['Product ID', 'Category']

calendar:
['Date', 'Seasonality', 'Holiday/Promotion']

inventory_snapshots:
['Date', 'Product ID', 'Inventory Level', 'Units Ordered']


### Identify Missing Required Fields

In [21]:
required_fields = {
    "sales_daily": [
        "date", "sku_id", "units_sold", "revenue",
        "unit_price", "promo_flag"
    ],
    
    "sku_master": [
        "sku_id", "category", "subcategory",
        "launch_date", "unit_cost", "list_price"
    ],
    
    "calendar": [
        "date", "week", "month", "season",
        "is_holiday", "promo_event"
    ],
    
    "inventory_snapshots": [
        "date", "sku_id", "on_hand_units",
        "on_order_units", "lead_time_days", "reorder_point"
    ]
}

available_columns = df.columns.tolist()

print("Available raw columns:")
print(available_columns)

print("\nRequired fields by table:")
for table, fields in required_fields.items():
    print(f"\n{table}:")
    print(fields)

Available raw columns:
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast', 'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion', 'Competitor Pricing', 'Seasonality']

Required fields by table:

sales_daily:
['date', 'sku_id', 'units_sold', 'revenue', 'unit_price', 'promo_flag']

sku_master:
['sku_id', 'category', 'subcategory', 'launch_date', 'unit_cost', 'list_price']

calendar:
['date', 'week', 'month', 'season', 'is_holiday', 'promo_event']

inventory_snapshots:
['date', 'sku_id', 'on_hand_units', 'on_order_units', 'lead_time_days', 'reorder_point']


## Create sales_daily Table

In [22]:
sales_daily = df[[
    "Date",
    "Product ID",
    "Units Sold",
    "Price",
    "Discount",
    "Holiday/Promotion"
]].copy()

sales_daily = sales_daily.rename(columns={
    "Date": "date",
    "Product ID": "sku_id",
    "Units Sold": "units_sold",
    "Price": "unit_price",
    "Discount": "discount_pct",
    "Holiday/Promotion": "promo_flag"
})

sales_daily["revenue"] = (
    sales_daily["units_sold"]
    * sales_daily["unit_price"]
    * (1 - sales_daily["discount_pct"] / 100)
)

sales_daily = sales_daily[
    ["date", "sku_id", "units_sold", "revenue", "unit_price",
     "discount_pct", "promo_flag"]
]

sales_daily.head()

,date,sku_id,units_sold,revenue,unit_price,discount_pct,promo_flag
0,2022-01-01,P0001,127,3403.600,33.50,20,0
1,2022-01-01,P0002,150,7561.200,63.01,20,0
2,2022-01-01,P0003,65,1637.415,27.99,10,1
3,2022-01-01,P0004,61,1796.328,32.72,10,1
4,2022-01-01,P0005,14,1030.960,73.64,0,0


### Validate sales_daily

In [23]:
print("Shape:", sales_daily.shape)

print(
    "Duplicate date-SKU rows:",
    sales_daily.duplicated(["date", "sku_id"]).sum()
)

print(
    "Missing values:",
    sales_daily.isnull().sum().sum()
)

print(
    "Negative revenue:",
    (sales_daily["revenue"] < 0).sum()
)

Shape: (73100, 7)
Duplicate date-SKU rows: 58480
Missing values: 0
Negative revenue: 0


In [24]:
sales_daily = (
    df.groupby(["Date", "Product ID"])
      .agg(
          units_sold=("Units Sold", "sum"),
          revenue=("Units Sold", lambda x: 0)  # temporary
      )
      .reset_index()
)

# Calculate revenue at transaction/store level first
df["revenue"] = (
    df["Units Sold"]
    * df["Price"]
    * (1 - df["Discount"] / 100)
)

sales_daily = (
    df.groupby(["Date", "Product ID"])
      .agg(
          units_sold=("Units Sold", "sum"),
          revenue=("revenue", "sum"),
          promo_flag=("Holiday/Promotion", "max"),
          discount_pct=("Discount", "mean")
      )
      .reset_index()
)

# Weighted average price based on units sold
weighted_price = (
    df.assign(price_value=df["Price"] * df["Units Sold"])
      .groupby(["Date", "Product ID"])["price_value"]
      .sum()
    /
    df.groupby(["Date", "Product ID"])["Units Sold"].sum()
)

sales_daily["unit_price"] = weighted_price.values

sales_daily = sales_daily.rename(columns={
    "Date": "date",
    "Product ID": "sku_id"
})

sales_daily = sales_daily[
    ["date", "sku_id", "units_sold", "revenue",
     "unit_price", "discount_pct", "promo_flag"]
]

sales_daily.head()

,date,sku_id,units_sold,revenue,unit_price,discount_pct,promo_flag
0,2022-01-01,P0001,726,23990.3725,37.457300,14.0,1
1,2022-01-01,P0002,773,36732.8190,52.402277,10.0,1
2,2022-01-01,P0003,732,35288.2790,58.617964,13.0,1
3,2022-01-01,P0004,455,18612.9920,47.587516,11.0,1
4,2022-01-01,P0005,716,54894.1030,89.597779,11.0,1


### Aggregation validate

In [25]:
print("Shape:", sales_daily.shape)

print(
    "Duplicate date-SKU rows:",
    sales_daily.duplicated(["date", "sku_id"]).sum()
)

print(
    "Missing values:",
    sales_daily.isnull().sum().sum()
)

print(
    "Negative revenue:",
    (sales_daily["revenue"] < 0).sum()
)

Shape: (14620, 7)
Duplicate date-SKU rows: 0
Missing values: 0
Negative revenue: 0


### Save sales_daily Table

In [26]:
sales_daily.to_csv(
    "../data/processed/sales_daily.csv",
    index=False
)

print("sales_daily.csv saved successfully.")

sales_daily.csv saved successfully.


## Create sku_master Table

In [27]:
sku_master = (
    df[["Product ID", "Category"]]
    .drop_duplicates()
    .rename(columns={
        "Product ID": "sku_id",
        "Category": "category"
    })
    .sort_values("sku_id")
    .reset_index(drop=True)
)

sku_master.head()

,sku_id,category
0,P0001,Groceries
1,P0001,Toys
2,P0001,Electronics
3,P0001,Clothing
4,P0001,Furniture


### Validate Product-Category Consistency

Check whether each Product ID maps consistently to a single category. This is necessary before creating the SKU master table.

In [28]:
product_category_check = (
    df.groupby("Product ID")["Category"]
      .nunique()
      .sort_values(ascending=False)
)

print(product_category_check)

Product ID
P0001    5
P0002    5
P0003    5
P0004    5
P0005    5
P0006    5
P0007    5
P0008    5
P0009    5
P0010    5
P0011    5
P0012    5
P0013    5
P0014    5
P0015    5
P0016    5
P0017    5
P0018    5
P0019    5
P0020    5
Name: Category, dtype: int64


### Create Unique SKU Identifier

In [29]:
df["sku_id"] = (
    df["Store ID"].astype(str)
    + "_"
    + df["Product ID"].astype(str)
)

print("Unique SKU IDs:", df["sku_id"].nunique())
print(df[["Store ID", "Product ID", "sku_id", "Category"]].head(10))

Unique SKU IDs: 100
  Store ID Product ID      sku_id     Category
0     S001      P0001  S001_P0001    Groceries
1     S001      P0002  S001_P0002         Toys
2     S001      P0003  S001_P0003         Toys
3     S001      P0004  S001_P0004         Toys
4     S001      P0005  S001_P0005  Electronics
5     S001      P0006  S001_P0006    Groceries
6     S001      P0007  S001_P0007    Furniture
7     S001      P0008  S001_P0008     Clothing
8     S001      P0009  S001_P0009  Electronics
9     S001      P0010  S001_P0010         Toys


### Create Unique sku_master

In [30]:
sku_master = (
    df[["sku_id", "Store ID", "Product ID", "Category"]]
    .drop_duplicates()
    .sort_values("sku_id")
    .reset_index(drop=True)
)

sku_master.head(10)

,sku_id,Store ID,Product ID,Category
0,S001_P0001,S001,P0001,Groceries
1,S001_P0001,S001,P0001,Electronics
2,S001_P0001,S001,P0001,Furniture
3,S001_P0001,S001,P0001,Toys
4,S001_P0001,S001,P0001,Clothing
5,S001_P0002,S001,P0002,Toys
6,S001_P0002,S001,P0002,Clothing
7,S001_P0002,S001,P0002,Electronics
8,S001_P0002,S001,P0002,Furniture
9,S001_P0002,S001,P0002,Groceries


### Investigate SKU Category Inconsistency

The same store-product SKU appears with multiple category labels in the source data. Quantify this inconsistency before deciding how the category field should be handled.

In [31]:
category_variation = (
    df.groupby("sku_id")["Category"]
      .nunique()
)

print("SKUs with multiple categories:",
      (category_variation > 1).sum())

print("Total unique SKUs:",
      category_variation.size)

SKUs with multiple categories: 100
Total unique SKUs: 100


### Handle Inconsistent Category Data

In [32]:
sku_master = (
    df[["sku_id", "Store ID", "Product ID"]]
    .drop_duplicates()
    .sort_values("sku_id")
    .reset_index(drop=True)
)

sku_master = sku_master.rename(columns={
    "Store ID": "store_id",
    "Product ID": "product_id"
})

print("Shape:", sku_master.shape)
print("Unique SKU IDs:", sku_master["sku_id"].nunique())

sku_master.head()

Shape: (100, 3)
Unique SKU IDs: 100


,sku_id,store_id,product_id
0,S001_P0001,S001,P0001
1,S001_P0002,S001,P0002
2,S001_P0003,S001,P0003
3,S001_P0004,S001,P0004
4,S001_P0005,S001,P0005


### Save sku_master Table

In [33]:
sku_master.to_csv(
    "../data/processed/sku_master.csv",
    index=False
)

print("sku_master.csv saved successfully.")

sku_master.csv saved successfully.


## Create calendar Table

Create a daily calendar table using the source date, seasonality, and promotion information. Calendar fields that are not directly supported by the source data will not be fabricated.

In [34]:
calendar = (
    df[["Date", "Seasonality", "Holiday/Promotion"]]
    .drop_duplicates()
    .copy()
)

calendar["week"] = calendar["Date"].dt.isocalendar().week.astype(int)
calendar["month"] = calendar["Date"].dt.month

calendar = calendar.rename(columns={
    "Date": "date",
    "Seasonality": "season",
    "Holiday/Promotion": "promo_event"
})

calendar = calendar[
    ["date", "week", "month", "season", "promo_event"]
].sort_values("date").reset_index(drop=True)

calendar.head()

,date,week,month,season,promo_event
0,2022-01-01,52,1,Autumn,0
1,2022-01-01,52,1,Summer,1
2,2022-01-01,52,1,Autumn,1
3,2022-01-01,52,1,Summer,0
4,2022-01-01,52,1,Winter,1


### Validate Date-Level Calendar Fields

In [35]:
date_consistency = df.groupby("Date").agg(
    unique_seasons=("Seasonality", "nunique"),
    unique_promo_flags=("Holiday/Promotion", "nunique")
)

print("Dates with multiple seasons:",
      (date_consistency["unique_seasons"] > 1).sum())

print("Dates with multiple promotion flags:",
      (date_consistency["unique_promo_flags"] > 1).sum())

Dates with multiple seasons: 731
Dates with multiple promotion flags: 731


### Handle Date-Level Inconsistencies

In [36]:
calendar = (
    df[["Date"]]
    .drop_duplicates()
    .copy()
)

calendar["week"] = calendar["Date"].dt.isocalendar().week.astype(int)
calendar["month"] = calendar["Date"].dt.month

calendar = calendar.rename(columns={
    "Date": "date"
})

calendar = calendar[
    ["date", "week", "month"]
].sort_values("date").reset_index(drop=True)

print("Shape:", calendar.shape)
calendar.head()

Shape: (731, 3)


,date,week,month
0,2022-01-01,52,1
1,2022-01-02,52,1
2,2022-01-03,1,1
3,2022-01-04,1,1
4,2022-01-05,1,1


In [37]:
print("Shape:", calendar.shape)
print("Duplicate dates:", calendar["date"].duplicated().sum())
print("Missing values:", calendar.isnull().sum().sum())
print("Date range:", calendar["date"].min(), "to", calendar["date"].max())

Shape: (731, 3)
Duplicate dates: 0
Missing values: 0
Date range: 2022-01-01 00:00:00 to 2024-01-01 00:00:00


### Save calendar Table

In [38]:
calendar.to_csv(
    "../data/processed/calendar.csv",
    index=False
)

print("calendar.csv saved successfully.")

calendar.csv saved successfully.


## Create inventory_snapshots Table

In [39]:
inventory_snapshots = (
    df[[
        "Date",
        "sku_id",
        "Inventory Level",
        "Units Ordered"
    ]]
    .copy()
    .rename(columns={
        "Date": "date",
        "Inventory Level": "on_hand_units",
        "Units Ordered": "on_order_units"
    })
)

inventory_snapshots.head()

,date,sku_id,on_hand_units,on_order_units
0,2022-01-01,S001_P0001,231,55
1,2022-01-01,S001_P0002,204,66
2,2022-01-01,S001_P0003,102,51
3,2022-01-01,S001_P0004,469,164
4,2022-01-01,S001_P0005,166,135


### Validate inventory_snapshots

In [40]:
print("Shape:", inventory_snapshots.shape)

print(
    "Duplicate date-SKU rows:",
    inventory_snapshots.duplicated(["date", "sku_id"]).sum()
)

print(
    "Missing values:",
    inventory_snapshots.isnull().sum().sum()
)

print(
    "Negative on-hand units:",
    (inventory_snapshots["on_hand_units"] < 0).sum()
)

print(
    "Negative on-order units:",
    (inventory_snapshots["on_order_units"] < 0).sum()
)

Shape: (73100, 4)
Duplicate date-SKU rows: 0
Missing values: 0
Negative on-hand units: 0
Negative on-order units: 0


### Save inventory_snapshots Table

In [41]:
inventory_snapshots.to_csv(
    "../data/processed/inventory_snapshots.csv",
    index=False
)

print("inventory_snapshots.csv saved successfully.")

inventory_snapshots.csv saved successfully.


In [42]:
from pathlib import Path

processed_path = Path("../data/processed")

for file in sorted(processed_path.glob("*.csv")):
    print(file.name)

calendar.csv
forecast_results.csv
inventory_snapshots.csv
model_comparison.csv
sales_daily.csv
sku_master.csv
sku_risk_results.csv
weekly_sales.csv


## Data Transformation Summary

The raw dataset contains observations at store-product-day level.

For FORESIGHT, Store ID and Product ID were combined to create a unique SKU identifier. Sales records were then aggregated from store-product-day level to SKU-day level.

Revenue was calculated using units sold, price, and discount. Unit price was represented using a units-sold weighted average at SKU-day level.

The four processed tables were created without fabricating unavailable source fields.

## Standardize Data Types and Column Names

In [43]:
# Make copies so the existing tables remain unchanged
sales_daily_clean = sales_daily.copy()
sku_master_clean = sku_master.copy()
calendar_clean = calendar.copy()
inventory_clean = inventory_snapshots.copy()

# Standardize date types
sales_daily_clean["date"] = pd.to_datetime(sales_daily_clean["date"])
calendar_clean["date"] = pd.to_datetime(calendar_clean["date"])
inventory_clean["date"] = pd.to_datetime(inventory_clean["date"])

# Standardize ID columns
sales_daily_clean["sku_id"] = sales_daily_clean["sku_id"].astype("string")

sku_master_clean["sku_id"] = sku_master_clean["sku_id"].astype("string")
sku_master_clean["store_id"] = sku_master_clean["store_id"].astype("string")
sku_master_clean["product_id"] = sku_master_clean["product_id"].astype("string")

inventory_clean["sku_id"] = inventory_clean["sku_id"].astype("string")

print("Data types standardized successfully.")

Data types standardized successfully.


## Validate Primary Keys

In [44]:
print("sales_daily duplicate keys:",
      sales_daily_clean.duplicated(["date", "sku_id"]).sum())

print("sku_master duplicate keys:",
      sku_master_clean.duplicated(["sku_id"]).sum())

print("calendar duplicate keys:",
      calendar_clean.duplicated(["date"]).sum())

print("inventory duplicate keys:",
      inventory_clean.duplicated(["date", "sku_id"]).sum())

sales_daily duplicate keys: 0
sku_master duplicate keys: 0
calendar duplicate keys: 0
inventory duplicate keys: 0


## Validate Table Relationships

In [45]:
master_skus = set(sku_master_clean["sku_id"])
calendar_dates = set(calendar_clean["date"])

sales_skus_missing = set(sales_daily_clean["sku_id"]) - master_skus
inventory_skus_missing = set(inventory_clean["sku_id"]) - master_skus

sales_dates_missing = set(sales_daily_clean["date"]) - calendar_dates
inventory_dates_missing = set(inventory_clean["date"]) - calendar_dates

print("SKUs missing from sku_master:", len(sales_skus_missing))
print("Inventory SKUs missing from sku_master:", len(inventory_skus_missing))
print("Sales dates missing from calendar:", len(sales_dates_missing))
print("Inventory dates missing from calendar:", len(inventory_dates_missing))

SKUs missing from sku_master: 20
Inventory SKUs missing from sku_master: 0
Sales dates missing from calendar: 0
Inventory dates missing from calendar: 0


In [46]:
sales_daily_clean["sku_id"] = (
    sales_daily_clean["sku_id"]
    .astype(str)
)

sales_daily_clean["sku_id"].head()

0    P0001
1    P0002
2    P0003
3    P0004
4    P0005
Name: sku_id, dtype: str

In [47]:
# Sales ko Store + Product level par rebuild karna

sales_base = df.copy()

sales_base["sku_id"] = (
    sales_base["Store ID"].astype(str)
    + "_"
    + sales_base["Product ID"].astype(str)
)

sales_base["revenue"] = (
    sales_base["Units Sold"]
    * sales_base["Price"]
    * (1 - sales_base["Discount"] / 100)
)

sales_daily_clean = (
    sales_base.groupby(["Date", "sku_id"])
    .agg(
        units_sold=("Units Sold", "sum"),
        revenue=("revenue", "sum"),
        unit_price=("Price", "mean"),
        discount_pct=("Discount", "mean"),
        promo_flag=("Holiday/Promotion", "max")
    )
    .reset_index()
)

sales_daily_clean = sales_daily_clean.rename(
    columns={"Date": "date"}
)

sales_daily_clean = sales_daily_clean[
    [
        "date",
        "sku_id",
        "units_sold",
        "revenue",
        "unit_price",
        "discount_pct",
        "promo_flag"
    ]
]

print("Shape:", sales_daily_clean.shape)
print(sales_daily_clean["sku_id"].head())

Shape: (73100, 7)
0    S001_P0001
1    S001_P0002
2    S001_P0003
3    S001_P0004
4    S001_P0005
Name: sku_id, dtype: str


In [48]:
master_skus = set(sku_master_clean["sku_id"])
calendar_dates = set(calendar_clean["date"])

sales_skus_missing = set(sales_daily_clean["sku_id"]) - master_skus
inventory_skus_missing = set(inventory_clean["sku_id"]) - master_skus

sales_dates_missing = set(sales_daily_clean["date"]) - calendar_dates
inventory_dates_missing = set(inventory_clean["date"]) - calendar_dates

print("SKUs missing from sku_master:", len(sales_skus_missing))
print("Inventory SKUs missing from sku_master:", len(inventory_skus_missing))
print("Sales dates missing from calendar:", len(sales_dates_missing))
print("Inventory dates missing from calendar:", len(inventory_dates_missing))

SKUs missing from sku_master: 0
Inventory SKUs missing from sku_master: 0
Sales dates missing from calendar: 0
Inventory dates missing from calendar: 0


In [49]:
sales_daily_clean.to_csv(
    "../data/processed/sales_daily.csv",
    index=False
)

print("sales_daily.csv saved successfully!")

sales_daily.csv saved successfully!


In [50]:
print("sales_daily:", sales_daily_clean.shape)
print("sku_master:", sku_master_clean.shape)
print("calendar:", calendar_clean.shape)
print("inventory_snapshots:", inventory_clean.shape)

print("\nFiles ready:")
print("✓ sales_daily.csv")
print("✓ sku_master.csv")
print("✓ calendar.csv")
print("✓ inventory_snapshots.csv")

sales_daily: (73100, 7)
sku_master: (100, 3)
calendar: (731, 3)
inventory_snapshots: (73100, 4)

Files ready:
✓ sales_daily.csv
✓ sku_master.csv
✓ calendar.csv
✓ inventory_snapshots.csv
